
# Proyecto — Data Stream Processor
## Contexto (extremadamente importante):
Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún commit he iniciar el proceso desde ese punto.

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.

## Objetivo
Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases ArrayStack y ArrayQueue proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

### En el método __init__ debe aparecer tres atributos:

Queue: registros que han llegado pero todavía no han sido procesados.

Stack: historial de cambios realizados, para poder deshacer los cambios más recientes.

Lista: como se encuentran los registros actualmente

Las clases ArrayStack y ArrayQueue ya están implementadas. No debe implementarlas nuevamente ni modificarlas.

In [1]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue

In [ ]:
from pyparsing import Empty


class DataProcesor:

    #   1. Se definen los tres atributos principales sin modificar la lógica interna de ArrayQueue ni ArrayStack.  
    
    def __init__(self):
        self._queue = ArrayQueue()
        self._stack = ArrayStack()
        self._state = []

     #  2. Se valida que el argumento cumpla con la estructura de 3 elementos (sensor, variable, value) y que el value sea un número.
     #  Lanza ValueError cuando no cumple estos parámetros. Si es correcto, lo añade a _queue sin modificar el estado ni el historial.   
    
    def add(self,record):
        if not isinstance(record, (tuple, list)) or len(record) != 3:
            raise ValueError(
                "El registro debe ser una tupla o lista de exactamente 3"
                " elementos: (sensor, variable, value)"
            )

        sensor, variable, value = record
        # Validar que el valor sea numérico
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise ValueError("El valor ('value') del registro debe ser numérico")

        # Agregar el registro a la cola respetando el orden de llegada
        self._queue.enqueue(record)

    #  3. Desencola el siguiente registro respetando el orden FIFO. Antes de actualizar _state, verifica si la combinación (sensor, variable) ya existía
    #   para guardar su valor previo en _stack. Si no existía, guarda en el historial un marcador indicando que la clave es nueva para que undo()
    #   pueda eliminarla si es necesario. Si la cola está vacía, genera la excepción Empty.
    
    def process_next(self):
        if self._queue.is_empty():
            raise Empty("No hay registros pendientes para procesar")

        record = self._queue.dequeue()
        sensor, variable, value = record

        previous_value = None
        existed_before = False
        index_to_update = -1

        for i, item in enumerate(self._state):
            if item[0] == sensor and item[1] == variable:
                existed_before = True
                previous_value = item[2]
                index_to_update = i
                break

        self._stack.push((sensor, variable, existed_before, previous_value))
        if existed_before:
            self._state[index_to_update] = (sensor, variable, value)
        else:
            self._state.append((sensor, variable, value))
        return record


# 4. Extrae del _stack (comportamiento LIFO) la información sobre el último registro modificado o creado.
# Si la variable existía antes de ser procesada, restaura la tupla (sensor, variable, previous_value) en _state.
# Si era una variable totalmente nueva que antes no existía en el sistema, la busca y la remueve de la lista _state.
# Lanza la excepción Empty si _stack está vacío
    
    def undo(self):
        if self._stack.is_empty():
            raise Empty("No hay cambios que deshacer en el historial")

        sensor, variable, existed_before, previous_value = self._stack.pop()

        if existed_before:
            for i, item in enumerate(self._state):
                if item[0] == sensor and item[1] == variable:
                    self._state[i] = (sensor, variable, previous_value)
                    break
        else:
            for i, item in enumerate(self._state):
                if item[0] == sensor and item[1] == variable:
                    self._state.pop(i)
                    break
    
# 5. Retorna la cantidad de elementos en _queue que aguardan ser procesados.
    def pending(self):
        return len(self._queue)

# 6.Recorre la lista _state buscando coincidencia exacta del par (sensor, variable) y retorna su value.
# Si la combinación no ha sido registrada/procesada, lanza KeyError.

    def current_value(self, sensor, variable):
        for item in self._state:
            if item[0] == sensor and item[1] == variable:
                return item[2]
            
        raise KeyError(
            f"No existe un valor para el sensor '{sensor}' y variable"
            f" '{variable}'"
        )

